# Post-process feedstock dissolution calculations from a batch of runs

Calculates how much of the applied rock has weathered using the model's mineral accounting (rather than inferred from CDR). `totals.Basalt.{Ca,Mg,K,Na}` is the cation inventory still locked in feedstock that has not dissolved; `totals.{Ca,Mg,K,Na}` is everything that was applied. Necessarily, they are equal at `t=0`. We can calculate the dissolution fraction on a cation charge basis by:

$$f_\text{diss}(t) = 1 - \frac{\text{locked}(t)}{\text{applied}(t)}$$

The calculation does not require a normative mineralogy, nor does it include losses like $\eta$ or other processes --- just dissolution!

We treat this outside of the other postprocess scripts because the other output zarrs don't need the basalt mineralogy pool at all.

Writes `postproc/<tag>/dissolution.zarr`, dims `cluster x feedstock x time`.

In [ ]:
import re, time, warnings
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np, pandas as pd, scipy.io as sio, s3fs, xarray as xr
from beerling_analysis import io

warnings.filterwarnings("ignore")

TAG        = "e0-conus-1gt"   # names the results/ prefix on S3
ROOT       = "s3://carbonplan-carbon-removal/ew-workflows-data/beerling-2025-model-extension"
OUT        = f"{ROOT}/postproc/{TAG}/dissolution.zarr"   # set to a local path to write locally
CACHE      = Path(io.CACHE) / TAG              # .mat staging; ~19 MB at a time per worker
KEEP_CACHE = False                             # False -> delete each .mat once read
WORKERS    = 12
RETRIES    = 5                                 # S3 drops a connection now and then

G2T = io.G_PER_M2_TO_T_PER_HA   # g/m2 -> t/ha
fs  = s3fs.S3FileSystem()
CACHE.mkdir(parents=True, exist_ok=True)

PAT = re.compile(r"^(?P<exp>E\d)_drv(?P<drv>\d+)_(?P<source>.+)_(?P<feedstock>[12])_(?P<tf>\d+)yr\.mat$")
runs = pd.DataFrame([
    {**PAT.match(n).groupdict(), "file": n}
    for n in (Path(k).name for k in fs.ls(f"{ROOT.split('://')[1]}/results/{TAG}"))
    if PAT.match(n)
]).astype({"drv": int, "feedstock": int, "tf": int})
runs["start_year"] = 2071 - runs.tf           # every deployment runs to 2070

assert len(runs), f"no runs under results/{TAG}"
assert (runs.groupby("drv").feedstock.nunique() == 2).all(), "a cluster is missing a mineralogy"
print(f"{TAG}  ->  {OUT}")
print(f"{len(runs)} runs, {runs.drv.nunique()} clusters, "
      f"start years {runs.start_year.min()}-{runs.start_year.max()}")

## The reduction

Both fields are on the 100/yr diag grid and are cumulative, so we reduce them to annual by the value at the end of the year. 

In [ ]:
CHG = dict(Ca=2., Mg=2., K=1., Na=1.)
done, t0 = [0], time.time()

def read_one(row):
    """(applied, locked) cation charge per simulated year, eq/m2, for one run."""
    p = CACHE / row.file
    for attempt in range(RETRIES):
        try:
            fs.get(f"{ROOT.split('://')[1]}/results/{TAG}/{row.file}", str(p))
            m = sio.loadmat(p, squeeze_me=True, struct_as_record=False,
                            variable_names=["totals"])   # skips diag's big arrays
            break
        except Exception as e:
            if attempt == RETRIES - 1:
                raise
            print(f"  retry {attempt + 1} {row.file}: {type(e).__name__}", flush=True)
            time.sleep(2 ** attempt)
        finally:
            if not KEEP_CACHE:
                p.unlink(missing_ok=True)

    t  = m["totals"]
    tf = int(round(float(np.atleast_1d(t.diag.t)[-1])))
    g  = lambda o, c: np.atleast_1d(getattr(o, c)).astype(float)
    applied = sum(w * g(t, c) for c, w in CHG.items())
    locked  = sum(w * g(t.Basalt, c) for c, w in CHG.items())
    end = lambda x: np.asarray(x, float)[100:100 * tf + 1:100]
    a, l = end(applied), end(locked)

    assert a.shape == (tf,) and l.shape == (tf,), f"{row.file}: the grid is not 100/yr"
    assert np.all(np.diff(a) >= -1e-9), f"{row.file}: applied charge is not monotonic"
    assert np.all(l <= a * (1 + 1e-9)), f"{row.file}: more locked than applied"

    done[0] += 1
    if done[0] % 50 == 0:
        el = time.time() - t0
        print(f"  {done[0]}/{len(runs)}  {el / 60:.1f} min  "
              f"eta {el / done[0] * (len(runs) - done[0]) / 60:.1f} min", flush=True)
    return a, l

with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    out = list(ex.map(read_one, [r for _, r in runs.iterrows()]))
print(f"read {len(out)} runs in {(time.time() - t0) / 60:.1f} min")

## Write, and check against `point.zarr`

The check against `point.zarr` confirms that what dissolved is at least what left the column plus precipitated as carbonate.

In [ ]:
clusters = np.array(sorted(runs.drv.unique()))
YEARS    = np.arange(runs.start_year.min(), 2071)
ci       = {d: i for i, d in enumerate(clusters)}
A = np.full((len(clusters), 2, len(YEARS)), np.nan)
L = np.full_like(A, np.nan)
for (_, r), (a, l) in zip(runs.iterrows(), out):
    j = int(np.where(YEARS == r.start_year)[0][0])
    A[ci[r.drv], r.feedstock - 1, j:j + len(a)] = a
    L[ci[r.drv], r.feedstock - 1, j:j + len(l)] = l

ds = xr.Dataset(
    dict(applied_eq=(("cluster", "feedstock", "time"), A),
         locked_eq=(("cluster", "feedstock", "time"), L)),
    coords=dict(cluster=clusters, feedstock=[1, 2], time=YEARS),
    attrs=dict(tag=TAG, source=f"{ROOT}/results/{TAG}",
               note="cation charge (2Ca+2Mg+K+Na) applied in the feedstock and still "
                    "locked in feedstock that has not dissolved, eq/m2, end of year, "
                    "per mineralogy cluster. Mix BOTH at w, then take "
                    "1 - locked/applied -- the quotient does not mix."))
for v in ("applied_eq", "locked_eq"):
    ds[v].attrs["units"] = "eq/m2"
ds.to_zarr(OUT, mode="w", consolidated=True)
print(f"wrote {OUT}  {dict(ds.sizes)}")

# -- against point.zarr: the mineral pool must cover export + precipitate ------
pt = xr.open_zarr(f"{ROOT}/postproc/{TAG}/point.zarr").load()

def per_point(v):
    a = (ds[v].sel(cluster=pt.cluster.values).drop_vars("cluster")
            .rename(cluster="point").assign_coords(point=pt.point))
    return pt.w * a.sel(feedstock=1) + (1 - pt.w) * a.sel(feedstock=2)

APPLIED   = per_point("applied_eq")
DISSOLVED = APPLIED - per_point("locked_eq")
# cdr_precip credits 1 mol CO2 per mol Ca precipitated, so it converts to 2 eq
ACCOUNTED = (pt.Na_export + pt.K_export + 2 * (pt.Mg_export + pt.Ca_export)
             + 2 * pt.cdr_precip / G2T / 44.009)
gap = ((DISSOLVED - ACCOUNTED) / APPLIED).values
ok  = np.isfinite(gap)
assert np.nanmin(gap) > -1e-3, "export + precipitate exceeds what the mineral pool says dissolved"

eqg = (APPLIED.sel(time=2070) / (pt.rock.sel(time=2070) / G2T)).values   # eq/g rock
print(f"feedstock charge {eqg.min():.5f}-{eqg.max():.5f} eq/g "
      f"({44.009 * eqg.min():.3f}-{44.009 * eqg.max():.3f} t CO2/t rock at full dissolution)")
f = (DISSOLVED / APPLIED).values[ok]
print(f"dissolved fraction over {ok.sum():,} point-years: "
      f"p5 {np.percentile(f, 5):.3f}  p50 {np.percentile(f, 50):.3f}  p95 {np.percentile(f, 95):.3f}")
print(f"in porewater or on the exchange, i.e. dissolved but neither exported nor "
      f"precipitated: median {np.median(gap[ok]):.1%} of applied charge, "
      f"max {np.max(gap[ok]):.1%}")
ds

In [ ]:
# ---